# Défi 2 NLP (medRxiv 2026) — Projet 3 : Recommandation d'articles
**Approche :** Machine Learning Classique (TF-IDF, BM25, Rocchio PRF & LSA)  
**Auteur :** Redouane Kenniche

---

### Objectif du notebook
Ce notebook met en place un moteur de recherche d'information classique combinant plusieurs représentations lexicales et sémantiques latentes (`TF-IDF`, `Okapi BM25`, expansion de requête de `Rocchio` et `LSA`) afin de recommander les 10 preprints les plus pertinents pour chaque requête, en optimisant les métriques **nDCG@10** et **MRR**.

## 1. Importation des bibliothèques et chargement des données
Chargement des modules nécessaires au calcul matriciel creux, à la vectorisation textuelle et à la réduction de dimension, suivi de l'ouverture du corpus, des requêtes d'entraînement et des jugements de pertinence (*qrels*).

In [7]:
import html
import os
import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize



CORPUS_FILE = trouver_fichier([
    "medrxiv2026_corpus.csv",
    "medrxiv2026_corpus.csv.gz",
    "medrxiv2026 corpus.csv",
])
QUERIES_TRAIN_FILE = trouver_fichier([
    "medrxiv2026_reco_queries_train.csv",
    "medrxiv2026_reco_queries_train.csv.gz",
    "medrxiv2026 reco queries train.csv",
])
QRELS_TRAIN_FILE = trouver_fichier([
    "medrxiv2026_reco_qrels_train.csv",
    "medrxiv2026_reco_qrels_train.csv.gz",
    "medrxiv2026 reco qrels train.csv",
])

corpus_df = pd.read_csv(CORPUS_FILE)
queries_df = pd.read_csv(QUERIES_TRAIN_FILE)
qrels_df = pd.read_csv(QRELS_TRAIN_FILE)

## 2. Nettoyage et préparation textuelle du corpus
Décodage des entités HTML et suppression ciblée des balises de mise en forme tout en conservant les opérateurs statistiques (ex. `p<0.05`). Le titre est répété deux fois dans `full_text` afin de renforcer le poids des concepts clés.

In [8]:
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    texte = html.unescape(texte).lower()
    # Suppression ciblée des balises HTML uniquement
    texte = re.sub(
        r"</?(?:p|b|i|u|sub|sup|br|em|strong|span|div|ul|ol|li)\b[^>]*>",
        " ",
        texte,
    )
    texte = re.sub(r"[^a-z0-9\s]", " ", texte)
    return re.sub(r"\s+", " ", texte).strip()


corpus_df["clean_title"] = corpus_df["title"].apply(nettoyer_texte)
corpus_df["clean_abstract"] = corpus_df["abstract"].apply(nettoyer_texte)
corpus_df["full_text"] = (
    corpus_df["clean_title"]
    + " "
    + corpus_df["clean_title"]
    + " "
    + corpus_df["clean_abstract"]
)
queries_df["clean_query"] = queries_df["query"].apply(nettoyer_texte)
doc_ids = corpus_df["ID"].values

## 3. Métriques d'évaluation et découpage Réglage / Validation
Implémentation des métriques officielles **nDCG@10** (critère principal) et **MRR** (critère de départage), puis séparation des requêtes en 80 % réglage (`TRAIN`) et 20 % validation (`DEV`).

In [9]:
TOP_K = 10


def dcg_at_k(relevances, k=10):
    relevances = np.asarray(relevances, dtype=float)[:k]
    if not relevances.size:
        return 0.0
    return np.sum(
        (2**relevances - 1) / np.log2(np.arange(2, relevances.size + 2))
    )


def ndcg_at_k(relevances, relevances_ideales, k=10):
    idcg = dcg_at_k(sorted(relevances_ideales, reverse=True), k)
    return dcg_at_k(relevances, k) / idcg if idcg else 0.0


def rr_at_k(relevances, k=10):
    for i, r in enumerate(relevances[:k]):
        if r > 0:
            return 1.0 / (i + 1)
    return 0.0


qrels_dict = {}
for q_id, d_id, rel in qrels_df[
    ["query_id", "ID", "relevance"]
].itertuples(index=False):
    qrels_dict.setdefault(q_id, {})[d_id] = rel


def evaluer(S, positions):
    nd, rr = [], []
    for i in positions:
        q_id = queries_df.loc[i, "query_id"]
        top = np.argsort(-S[i], kind="stable")[:TOP_K]
        rels = [qrels_dict[q_id].get(d, 0) for d in doc_ids[top]]
        nd.append(ndcg_at_k(rels, list(qrels_dict[q_id].values()), TOP_K))
        rr.append(rr_at_k(rels, TOP_K))
    return {
        "nDCG@10": round(float(np.mean(nd)), 4),
        "MRR": round(float(np.mean(rr)), 4),
    }


# Découpage 80 % réglage / 20 % validation
rng = np.random.RandomState(42)
perm = rng.permutation(len(queries_df))
n_dev = int(0.2 * len(perm))
DEV, TRAIN = np.sort(perm[:n_dev]), np.sort(perm[n_dev:])

## 4. Calcul des matrices de similarité (TF-IDF, BM25, Rocchio, LSA & Caractères)
Construction de cinq signaux complémentaires de pertinence :
1. **TF-IDF** (unigrammes et bigrammes) ;
2. **Okapi BM25** avec optimisation de grille sur les hyperparamètres $k_1$ et $b$ ;
3. **Expansion de requête de Rocchio** (*Pseudo-Relevance Feedback*) sur les 3 meilleurs résultats BM25 ;
4. **LSA** (`TruncatedSVD` à 150 composantes) pour capturer la sémantique latente ;
5. **N-grammes de caractères** (3 à 5 caractères) pour la robustesse morphologique.

In [10]:
vectorizer = TfidfVectorizer(
    stop_words="english", ngram_range=(1, 2), min_df=2, sublinear_tf=True
)
corpus_vectors = vectorizer.fit_transform(corpus_df["full_text"])
query_vectors = vectorizer.transform(queries_df["clean_query"])
S_tfidf = (query_vectors @ corpus_vectors.T).toarray()


class BM25:

    def __init__(self):
        self.cv = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)

    def fit(self, docs):
        tf = self.cv.fit_transform(docs).tocsr().astype(np.float32)
        N = tf.shape[0]
        df = np.bincount(tf.indices, minlength=tf.shape[1])
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        self.tf = tf.tocoo()
        self.dl = np.asarray(tf.sum(1)).ravel()
        self.avgdl = self.dl.mean()
        return self

    def set_params(self, k1=1.2, b=0.75):
        t = self.tf
        denom = t.data + k1 * (1 - b + b * self.dl[t.row] / self.avgdl)
        data = t.data * (k1 + 1) / denom * self.idf[t.col]
        self.W = sp.csr_matrix((data, (t.row, t.col)), shape=t.shape)
        return self

    def scores(self, queries_clean):
        Q = self.cv.transform(queries_clean).tocsr()
        Q.data[:] = 1.0
        return (Q @ self.W.T).toarray()


bm25 = BM25().fit(corpus_df["full_text"])
best_bm25 = (-1, 1.2, 0.75)
for k1 in [0.9, 1.2, 1.5]:
    for b in [0.6, 0.75, 0.9]:
        bm25.set_params(k1, b)
        sc = evaluer(bm25.scores(queries_df["clean_query"]), TRAIN)["nDCG@10"]
        if sc > best_bm25[0]:
            best_bm25 = (sc, k1, b)

BM_K1, BM_B = best_bm25[1], best_bm25[2]
bm25.set_params(BM_K1, BM_B)
S_bm25 = bm25.scores(queries_df["clean_query"])


# Expansion de requête par pseudo-relevance feedback (Rocchio)
def rocchio(S_premier_tour, Q, X, k=3, beta=0.5):
    nq = S_premier_tour.shape[0]
    top = np.argpartition(-S_premier_tour, k - 1, axis=1)[:, :k]
    sel = sp.csr_matrix(
        (np.full(nq * k, 1.0 / k), (np.repeat(np.arange(nq), k), top.ravel())),
        shape=(nq, X.shape[0]),
    )
    Q2 = normalize(Q + beta * (sel @ X))
    return (Q2 @ X.T).toarray()


S_prf = rocchio(S_bm25, query_vectors, corpus_vectors, k=3, beta=0.5)

# Réduction de dimension sémantique latente (LSA)
lsa = TruncatedSVD(n_components=150, random_state=42)
doc_lsa = normalize(lsa.fit_transform(corpus_vectors))
q_lsa = normalize(lsa.transform(query_vectors))
S_lsa = q_lsa @ doc_lsa.T

# N-grammes de caractères
char_vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=5, sublinear_tf=True
)
doc_char = char_vec.fit_transform(corpus_df["full_text"])
q_char = char_vec.transform(queries_df["clean_query"])
S_char = (q_char @ doc_char.T).toarray()

## 5. Fusion pondérée des scores et évaluation
Normalisation par requête des matrices de scores et recherche des poids optimaux sur le jeu de réglage (`TRAIN`), puis vérification de la généralisation sur le jeu de validation (`DEV`).

In [11]:
def norm_row(M):
    m_max = np.maximum(M.max(axis=1, keepdims=True), 1e-9)
    return M / m_max


S_bm25_n, S_prf_n, S_lsa_n, S_char_n = (
    norm_row(S_bm25),
    norm_row(S_prf),
    norm_row(S_lsa),
    norm_row(S_char),
)

best_fusion = (-1, (0.5, 0.25, 0.15, 0.10))
for w_bm in [0.4, 0.5, 0.6]:
    for w_prf in [0.15, 0.25]:
        for w_lsa in [0.10, 0.20]:
            w_ch = round(1.0 - w_bm - w_prf - w_lsa, 2)
            if w_ch < 0:
                continue
            S_comb = (
                w_bm * S_bm25_n
                + w_prf * S_prf_n
                + w_lsa * S_lsa_n
                + w_ch * S_char_n
            )
            sc = evaluer(S_comb, TRAIN)["nDCG@10"]
            if sc > best_fusion[0]:
                best_fusion = (sc, (w_bm, w_prf, w_lsa, w_ch))

W_BM, W_PRF, W_LSA, W_CH = best_fusion[1]
S_final_ml = (
    W_BM * S_bm25_n + W_PRF * S_prf_n + W_LSA * S_lsa_n + W_CH * S_char_n
)

print(
    "Fusion ML - Réglage :",
    evaluer(S_final_ml, TRAIN),
    "| Validation :",
    evaluer(S_final_ml, DEV),
)

Fusion ML - Réglage : {'nDCG@10': 0.498, 'MRR': 0.6898} | Validation : {'nDCG@10': 0.4914, 'MRR': 0.7492}


## 6. Génération des recommandations sur le jeu de test
Calcul du classement Top-10 pour chacune des requêtes du jeu de test officiel et export au format CSV attendu `(query_id, ID, rank)`.

In [12]:
def generer_predictions_test_ml(
    fichier_test, fichier_sortie="Défi2_Projet3_ML.csv"
):
    test_q_df = pd.read_csv(fichier_test)
    clean_q = test_q_df["query"].apply(nettoyer_texte)

    qv_tfidf = vectorizer.transform(clean_q)
    s_bm = norm_row(bm25.scores(clean_q))
    s_pr = norm_row(rocchio(s_bm, qv_tfidf, corpus_vectors, k=3, beta=0.5))
    s_ls = norm_row(normalize(lsa.transform(qv_tfidf)) @ doc_lsa.T)
    s_ch = norm_row((char_vec.transform(clean_q) @ doc_char.T).toarray())

    S_test = W_BM * s_bm + W_PRF * s_pr + W_LSA * s_ls + W_CH * s_ch

    rows = []
    for i, q_id in enumerate(test_q_df["query_id"]):
        top_idx = np.argsort(-S_test[i], kind="stable")[:TOP_K]
        for rank, doc_idx in enumerate(top_idx, start=1):
            rows.append(
                {"query_id": q_id, "ID": doc_ids[doc_idx], "rank": rank}
            )

    pd.DataFrame(rows).to_csv(fichier_sortie, index=False)
    print(f"Prédictions ML générées : {fichier_sortie} ({len(rows)} lignes)")


for test_candidate in [
    "medrxiv2026 reco queries test.csv",
    "medrxiv2026_reco_queries_test.csv",
]:
    if os.path.exists(test_candidate):
        generer_predictions_test_ml(
            test_candidate, fichier_sortie="Défi2_Projet3_ML.csv"
        )
        break

In [14]:
for test_candidate in [
    "medrxiv2026 reco queries test.csv",
    "medrxiv2026_reco_queries_test.csv",
    "medrxiv2026 reco queries test.csv.gz",
    "medrxiv2026_reco_queries_test.csv.gz",
]:
    if os.path.exists(test_candidate):
        generer_predictions_test_ml(
            test_candidate, fichier_sortie="Défi2_Projet3_ML.csv"
        )
        break

Prédictions ML générées : Défi2_Projet3_ML.csv (2000 lignes)
